# Chapter 5: Loading and Saving Data

The programs from Chapter 5 of *Data Analysis with Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name.

Run the cells from top to bottom (**Run > Run All Cells**). The programs read the datasets from `../data/`, so keep this notebook in its chapter folder.

## Reading CSV Files Carefully

`read_options.py`

In [1]:
import pandas as pd

orders = pd.read_csv(
    "../data/orders.csv",
    usecols=["order_id", "order_date", "region", "category", "quantity"],
    parse_dates=["order_date"],
    dtype={"region": "category", "category": "category"},
)
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 5833 entries, 0 to 5832
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    5833 non-null   str           
 1   order_date  5833 non-null   datetime64[us]
 2   region      5833 non-null   category      
 3   category    5833 non-null   category      
 4   quantity    5833 non-null   int64         
dtypes: category(2), datetime64[us](1), int64(1), str(1)
memory usage: 148.6 KB


## Excel Workbooks

`excel_roundtrip.py`

In [2]:
import pandas as pd

products = pd.read_csv("../data/products.csv")
customers = pd.read_csv("../data/customers.csv")

# One workbook, two sheets
with pd.ExcelWriter("shop.xlsx") as writer:
    products.to_excel(writer, sheet_name="Products", index=False)
    customers.to_excel(writer, sheet_name="Customers", index=False)

sheets = pd.read_excel("shop.xlsx", sheet_name=None)  # None: every sheet
for name, table in sheets.items():
    print(f"{name}: {table.shape[0]} rows, {table.shape[1]} columns")
print(sheets["Products"].head(3))

Products: 15 rows, 4 columns
Customers: 1200 rows, 5 columns
         product category  list_price  unit_cost
0     Chef Knife  Kitchen          45      21.46
1  Cast Iron Pan  Kitchen          60      32.76
2        Mug Set  Kitchen          24      12.48


## JSON

`json_io.py`

In [3]:
import json

import pandas as pd

products = pd.read_csv("../data/products.csv")
products.head(2).to_json("products.json", orient="records", indent=2)
print(open("products.json").read())

# Nested JSON, as many web services return it, flattened into columns
reply = json.loads("""
[{"id": 1, "customer": {"name": "Asha", "city": "Pune"}, "total": 42.5},
 {"id": 2, "customer": {"name": "Ben", "city": "Leeds"}, "total": 18.0}]
""")
print(pd.json_normalize(reply))

[
  {
    "product":"Chef Knife",
    "category":"Kitchen",
    "list_price":45.0,
    "unit_cost":21.46
  },
  {
    "product":"Cast Iron Pan",
    "category":"Kitchen",
    "list_price":60.0,
    "unit_cost":32.76
  }
]
   id  total customer.name customer.city
0   1   42.5          Asha          Pune
1   2   18.0           Ben         Leeds


## SQL Databases

`sqlite_io.py`

In [4]:
import sqlite3

import pandas as pd

orders = pd.read_csv("../data/orders.csv")
with sqlite3.connect("shop.db") as conn:
    orders.to_sql("orders", conn, if_exists="replace", index=False)
    top = pd.read_sql(
        """SELECT region, COUNT(*) AS orders, SUM(quantity) AS units
           FROM orders GROUP BY region ORDER BY units DESC""",
        conn,
    )
print(top)

  region  orders  units
0  North    1768   2797
1   East    1560   2497
2  South    1300   2123
3   West    1205   1970
